# 댓글 분석 노트북 입니다.

## 0. 라이브러리 불러오기

In [1]:
from dotenv import load_dotenv
import db_utils
import torch
from importlib import reload
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sqlalchemy import text

/Users/sj.kang/Desktop/project2/comment_analysis/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. DB 연결

In [2]:
db_utils.test_connection()

DB 연결 성공
host     : 127.0.0.1:3306
database : youtube_model_db


In [3]:
load_dotenv("../../.env", override=True)

# 예시: 특정 채널(또는 비디오)의 댓글을 최대 1000개 가져옵니다.
query = """
    SELECT comment_id, channel_identifier, text, like_count 
    FROM comments_table 
    WHERE text IS NOT NULL AND text != ''
"""
comments_df = db_utils.run_query(query, display=True)

print(f"불러온 데이터 개수: {len(comments_df)}")


,comment_id,channel_identifier,text,like_count
0,Ugg-AWukg_ewoHgCoAEC,CH04929,헬로비너스 넘 나좋은것 쭉 응원 할게요 헬로비너스 화이팅,5
1,Ugg3tkIKBjovwngCoAEC,CH04929,노래 좋은데 왜 안뜨지..,2
2,Ugg7otrBa9ZRwXgCoAEC,CH04929,RISE GIRL ❤❤❤❤❤❤,3
3,Ugg8Dgr63iWAYngCoAEC,CH04929,I love you so much Aengie!! 💓,1
4,Ugg8lCGoPoJZZXgCoAEC,CH04929,今はTWICE,0
...,...,...,...,...
79932,UgzzzxnucsTYesRuTgR4AaABAg,CH07239,요즘 왜 라이브 유튜브로 안 올라오나요?,7
79933,UgzzZxOn2J0G1Dw9nwd4AaABAg,CH06576,형님 일주일만입니다. 기다리고 있었습니다,2
79934,UgzzzyQZHo0vZ0ydPi94AaABAg,CH03630,우왕~~~댓글 1등이에요 하늬님 미국에서 잘 살고있는거같아 맘이 놓이네요,7
79935,UgzzzzkBdE0qzBvoMI14AaABAg,CH07008,예전부터 느낀게 조혜련 저렇게 화사하게 염색하고 조혜련 이경실 머리 저렇게 여성스럽...,13


불러온 데이터 개수: 79937


## 2. 데이터 전처리

In [4]:
# 결측치 제거
comments_df = comments_df.dropna(subset=['text']).reset_index(drop=True)

print(f"불러온 데이터 개수: {len(comments_df)}")

불러온 데이터 개수: 79937


In [5]:
# 필요하다면 여기에 정규표현식을 활용한 특수문자 제거 로직을 추가할 수 있습니다.
texts = comments_df['text'].tolist()

## 3. 모델 로드 및 설정(KcELECTRA)

In [6]:
device = torch.device('mps' if torch.mps.is_available() else 'cpu')
device


device(type='mps')

In [7]:
MODEL_NAME = "beomi/KcELECTRA-base-v2022"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.to(device)
model.eval() # 추론 모드 전환


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 43885.59it/s]
[transformers] ElectraForSequenceClassification LOAD REPORT from: beomi/KcELECTRA-base-v2022
Key                                               | Status     | 
--------------------------------------------------+------------+-
discriminator_predictions.dense_prediction.bias   | UNEXPECTED | 
discriminator_predictions.dense.weight            | UNEXPECTED | 
discriminator_predictions.dense_prediction.weight | UNEXPECTED | 
discriminator_predictions.dense.bias              | UNEXPECTED | 
classifier.dense.weight                           | MISSING    | 
classifier.out_proj.weight                        | MISSING    | 
classifier.dense.bias                             | MISSING    | 
classifier.out_proj.bias                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because m

ElectraForSequenceClassification(
  (electra): ElectraModel(
    (embeddings): ElectraEmbeddings(
      (word_embeddings): Embedding(54343, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): ElectraEncoder(
      (layer): ModuleList(
        (0-11): 12 x ElectraLayer(
          (attention): ElectraAttention(
            (self): ElectraSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): ElectraSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (La

### 3.1. 추론

In [8]:

# ---------------------------------------------------------
# 4. 추론 (Inference)
# ---------------------------------------------------------
predictions = []
probabilities = []

# 배치 사이즈 설정 (메모리에 맞게 조절)
batch_size = 16

for i in range(0, len(texts), batch_size):
    batch_texts = texts[i:i + batch_size]
    
    # 토큰화
    inputs = tokenizer(
        batch_texts, 
        return_tensors="pt", 
        padding=True, 
        truncation=True, 
        max_length=128
    ).to(device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        # Softmax를 거쳐 확률값 도출
        probs = torch.nn.functional.softmax(logits, dim=-1)
        
        # 가장 높은 확률을 가진 클래스(Label) 추출
        preds = torch.argmax(probs, dim=-1)
        
        predictions.extend(preds.cpu().numpy())
        # 긍정 확률(예: index 1)만 저장할 경우
        probabilities.extend(probs[:, 1].cpu().numpy())


### 3.2. 분석결과 병합

In [9]:

# ---------------------------------------------------------
# 5. 분석 결과 병합 및 적재
# ---------------------------------------------------------
# 예측 라벨과 확률을 원본 DataFrame에 병합
comments_df['sentiment_label'] = predictions
comments_df['sentiment_prob'] = probabilities

# 라벨 매핑 예시 (0: 부정, 1: 긍정)
# label_map = {0: "부정", 1: "긍정"}
# comments_df['sentiment_text'] = comments_df['sentiment_label'].map(label_map)

# 결과 확인
comments_df[['comment_id', 'text', 'sentiment_label', 'sentiment_prob']].tail(50)

# DB에 분석 결과 업데이트 (comment_id를 키값으로 사용)
# db_utils.update_sentiment_results(comments_df, table_name='comments_table')

# (선택) 분석 결과를 CSV로 저장합니다.
# comments_df.to_csv("comment_sentiment_results.csv", index=False, encoding="utf-8-sig")

,comment_id,text,sentiment_label,sentiment_prob
79887,UgzzXVWol77Su3BxANF4AaABAg,Awww mi Ha Ji Won 😍,1,0.521315
79888,UgzzXwkQzQUn9NaUT_x4AaABAg,으엉 진짜 오랜만이당...ㅜㅜㅜ,1,0.524276
79889,UgzzXYdiBiZx73PfBJp4AaABAg,아쉽지만 누구보다 빛났던 무대✨ 하루 님 정말 최고였습니다 👏,1,0.525910
79890,UgzZxzTJfyBXvJm5vCV4AaABAg,우리 아이 7살 때 처음 자라다 갔을 때 선생님이에요. 너무 반갑다. 경표 선생님!!,0,0.488275
79891,UgzzxZxvWy-QTcgFFCd4AaABAg,I remember when I first learnt about Jake and ...,1,0.509791
79892,UgzZY_vlcH1Wf7K1jmp4AaABAg,점점 망해가네 하… 내 최애프로였는데 ㅠ,0,0.490995
79893,UgzZy6bJRMoeIZuwhPp4AaABAg,카멜보드 저도 있는데! 유용하더라고요 ˃ᴗ˂,1,0.513409
79894,UgzzY6YfROVnyxsrgth4AaABAg,꺅 계원예대 꼭 와주세요 목 빠지게 기다려요!!,1,0.511744
79895,UgzzY8zfDJwbCGeR0TR4AaABAg,하임이 송혜교 느낌 나네 귀여워 ❤❤❤,0,0.481387
79896,UgzzyCRB6NPGfkpzXfF4AaABAg,17:01 걱정마세요 용국씨 밥은 아마 1년 안에는 절대 못 먹을거에요ㅋㅋㅋㅋㅋㅋㅌ...,1,0.511045


In [31]:
comments_df.to_csv("comments_df.csv")

In [30]:
import importlib
import db_utils



# 2. 함수를 호출합니다. 이제 내부적으로 10,000개씩 나누어 처리합니다.
print("데이터 적재 시작...")
#db_utils.update_sentiment_results(comments_df, table_name="comments_table")

# 만약 10,000개도 너무 많아 끊긴다면, chunksize를 더 작게 조절할 수 있습니다.
db_utils.update_sentiment_results(comments_df, table_name="comments_table", chunksize=1000)


데이터 적재 시작...


TypeError: update_sentiment_results() got an unexpected keyword argument 'chunksize'

In [10]:

# 1. DB 연결 엔진 가져오기
engine = db_utils.get_engine()

# DB_NAME을 가져옵니다. (db_utils.DB_CONFIG 활용)
db_name = db_utils.DB_CONFIG['database']
table_name = 'comments_table'
column_name = 'sentiment_label'


In [11]:

# ---------------------------------------------------------
# Step 1. comments_table에 sentiment_label 컬럼이 없으면 추가
# ---------------------------------------------------------
with engine.begin() as conn:
    # 컬럼이 존재하는지 확인하는 쿼리 (MySQL/MariaDB 전용)
    check_col_query = text(f"""
        SELECT COUNT(*)
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = '{db_name}'
          AND TABLE_NAME = '{table_name}'
          AND COLUMN_NAME = '{column_name}'
    """)
    
    # 결과가 0이면 컬럼이 없다는 뜻
    column_exists = conn.execute(check_col_query).scalar()
    
    if column_exists == 0:
        print(f"'{table_name}'에 '{column_name}' 컬럼이 없어서 새로 추가합니다...")
        # 데이터 타입은 라벨 형태(예: '긍정', '부정')에 맞게 VARCHAR로 설정했습니다.
        # 만약 숫자형 라벨(0, 1, 2)이라면 INT로 변경하셔도 됩니다.
        add_col_query = text(f"ALTER TABLE {table_name} ADD COLUMN {column_name} VARCHAR(50);")
        conn.execute(add_col_query)
    else:
        print(f"'{table_name}'에 '{column_name}' 컬럼이 이미 존재합니다.")


'comments_table'에 'sentiment_label' 컬럼이 이미 존재합니다.


In [12]:

# ---------------------------------------------------------
# Step 2. 임시 테이블을 생성하여 데이터 업로드
# ---------------------------------------------------------
update_df = comments_df[['comment_id', 'sentiment_label']]
temp_table_name = 'temp_comments_sentiment'

print(f"데이터베이스에 임시 테이블({temp_table_name})을 업로드 중...")
update_df.to_sql(name=temp_table_name, con=engine, if_exists='replace', index=False)



데이터베이스에 임시 테이블(temp_comments_sentiment)을 업로드 중...


79937

In [25]:
# ---------------------------------------------------------
# Step 3. INNER JOIN을 사용하여 실제 테이블 업데이트 및 정리
# ---------------------------------------------------------
print("분석 결과를 실제 테이블에 적재(업데이트) 중...")
with engine.begin() as conn:
    # 쉼표(,) 플랫 조인 형태로 쿼리를 변경하고 세미콜론을 제거합니다.
    update_query = text(f"""           
        UPDATE {table_name} AS c, {temp_table_name} AS t
        SET c.{column_name} = t.{column_name}
        WHERE c.comment_id = t.comment_id
    """)
    
    # 쿼리 실행
    conn.execute(update_query)
    
    # 임시 테이블 삭제
    conn.execute(text(f"DROP TABLE {temp_table_name}"))

print("🎉 데이터베이스 적재가 성공적으로 완료되었습니다!")

분석 결과를 실제 테이블에 적재(업데이트) 중...


OperationalError: (pymysql.err.OperationalError) (1267, "Illegal mix of collations (utf8mb4_unicode_ci,IMPLICIT) and (utf8mb4_0900_ai_ci,IMPLICIT) for operation '='")
[SQL:            
        UPDATE comments_table AS c, temp_comments_sentiment AS t
        SET c.sentiment_label = t.sentiment_label
        WHERE c.comment_id = t.comment_id
    ]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
import db_utils
# 데이터프레임(comments_df)에 담긴 분석 결과를 DB에 적재합니다.
# 컬럼이 없다면 내부적으로 알아서 생성하고, 임시 테이블을 통한 업데이트와 뒷정리까지 처리합니다.
db_utils.update_sentiment_results(comments_df, table_name="comments_table")

print("분석 결과 데이터베이스 적재 완료!")